# Day 2: MCMC Methods and Generalized Linear Models

**Applied Bayesian Statistics Workshop**

---

## Learning Objectives

1. Build a Metropolis-Hastings sampler from scratch
2. Understand the NUTS sampler in PyMC
3. Diagnose MCMC convergence with trace plots, R-hat, ESS, and divergences
4. Perform posterior predictive checks
5. Fit Bayesian logistic regression on synthetic data
6. Fit Bayesian Poisson regression for count data

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import pymc as pm
import arviz as az

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.size"] = 12
az.style.use("arviz-darkgrid")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 1. MCMC Intuition: Metropolis-Hastings from Scratch

The Metropolis-Hastings algorithm constructs a Markov chain whose stationary
distribution is the target posterior. At each step:

1. Propose a new value $\theta^* \sim q(\theta^* \mid \theta_t)$
2. Compute the acceptance ratio: $r = \frac{P(\theta^* \mid \text{data})}{P(\theta_t \mid \text{data})}$
3. Accept with probability $\min(1, r)$; otherwise stay at $\theta_t$

We'll sample from a posterior for the coin-flip problem: $\theta \mid k, N$.

In [ ]:
def log_posterior(theta, k, N, alpha_prior=1, beta_prior=1):
    """Log-posterior for Beta-Binomial model (unnormalized)."""
    if theta <= 0 or theta >= 1:
        return -np.inf
    log_prior = stats.beta.logpdf(theta, alpha_prior, beta_prior)
    log_lik = stats.binom.logpmf(k, N, theta)
    return log_prior + log_lik


def metropolis_hastings(log_post_fn, initial, n_samples, proposal_sd,
                        rng, **kwargs):
    """Simple Metropolis-Hastings sampler with Gaussian proposal."""
    samples = np.zeros(n_samples)
    samples[0] = initial
    current_log_p = log_post_fn(initial, **kwargs)
    n_accepted = 0

    for i in range(1, n_samples):
        # Propose
        proposal = samples[i - 1] + rng.normal(0, proposal_sd)
        proposal_log_p = log_post_fn(proposal, **kwargs)

        # Acceptance ratio (log scale)
        log_ratio = proposal_log_p - current_log_p

        # Accept / reject
        if np.log(rng.uniform()) < log_ratio:
            samples[i] = proposal
            current_log_p = proposal_log_p
            n_accepted += 1
        else:
            samples[i] = samples[i - 1]

    acceptance_rate = n_accepted / (n_samples - 1)
    return samples, acceptance_rate


# Run the sampler
k_obs, N_obs = 14, 20

samples_mh, acc_rate = metropolis_hastings(
    log_posterior, initial=0.5, n_samples=10000, proposal_sd=0.1,
    rng=rng, k=k_obs, N=N_obs, alpha_prior=2, beta_prior=2
)

print(f"Acceptance rate: {acc_rate:.1%}")

In [ ]:
# --- Visualize the MH chain ---
burn_in = 1000
samples_post_burn = samples_mh[burn_in:]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Trace
axes[0].plot(samples_mh, lw=0.5, alpha=0.7)
axes[0].axvline(burn_in, color="red", ls="--", label="Burn-in cutoff")
axes[0].set_title("Trace Plot")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel(r"$\theta$")
axes[0].legend()

# Histogram vs analytical
x_grid = np.linspace(0, 1, 200)
# Analytical posterior: Beta(2+14, 2+6) = Beta(16, 8)
axes[1].hist(samples_post_burn, bins=50, density=True, alpha=0.6,
             color="steelblue", edgecolor="white", label="MH samples")
axes[1].plot(x_grid, stats.beta.pdf(x_grid, 16, 8), "r-", lw=2.5,
             label="Analytical Beta(16,8)")
axes[1].set_title("Posterior Distribution")
axes[1].set_xlabel(r"$\theta$")
axes[1].legend()

# Autocorrelation
max_lag = 100
acf = np.correlate(samples_post_burn - samples_post_burn.mean(),
                   samples_post_burn - samples_post_burn.mean(), mode="full")
acf = acf[len(acf)//2:]
acf = acf / acf[0]
axes[2].bar(range(max_lag), acf[:max_lag], color="steelblue", alpha=0.7)
axes[2].set_title("Autocorrelation")
axes[2].set_xlabel("Lag")
axes[2].set_ylabel("ACF")

plt.tight_layout()
plt.show()

### Effect of Proposal Scale

The proposal standard deviation $\sigma_q$ controls the trade-off between
exploration (large steps) and acceptance rate (small steps).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, prop_sd, label in zip(axes, [0.005, 0.1, 1.0],
                                ["Too small", "Just right", "Too large"]):
    s, ar = metropolis_hastings(
        log_posterior, initial=0.5, n_samples=5000, proposal_sd=prop_sd,
        rng=np.random.default_rng(42), k=k_obs, N=N_obs,
        alpha_prior=2, beta_prior=2
    )
    ax.plot(s[:2000], lw=0.5)
    ax.set_title(f"{label} ($\\sigma_q$={prop_sd}, accept={ar:.0%})")
    ax.set_xlabel("Iteration")
    ax.set_ylabel(r"$\theta$")
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 2. NUTS Sampler in PyMC

The **No-U-Turn Sampler (NUTS)** is an adaptive variant of Hamiltonian Monte Carlo.
It uses gradient information to propose distant states with high acceptance probability,
eliminating the need to hand-tune proposal scales.

PyMC uses NUTS by default.

In [ ]:
# --- NUTS via PyMC: Normal model ---
# Generate data from a known Normal distribution
true_mu, true_sigma = 3.0, 1.5
data = rng.normal(true_mu, true_sigma, size=50)

with pm.Model() as normal_model:
    mu = pm.Normal("mu", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=10)
    obs = pm.Normal("obs", mu=mu, sigma=sigma, observed=data)

    # NUTS is the default sampler for continuous variables
    trace_normal = pm.sample(
        2000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        return_inferencedata=True
    )

print(az.summary(trace_normal, var_names=["mu", "sigma"]))

## 3. MCMC Diagnostics

Key diagnostics:
- **Trace plots**: visual inspection of chain mixing
- **R-hat**: measures between-chain vs within-chain variance (should be < 1.01)
- **ESS** (Effective Sample Size): accounts for autocorrelation (aim for > 400)
- **Divergences**: indicate the sampler struggled with the posterior geometry

In [ ]:
# --- Trace plots ---
az.plot_trace(trace_normal, var_names=["mu", "sigma"])
plt.tight_layout()
plt.show()

In [ ]:
# --- R-hat and ESS ---
summary = az.summary(trace_normal, var_names=["mu", "sigma"])
print("\n--- Convergence Diagnostics ---")
print(summary[["r_hat", "ess_bulk", "ess_tail"]])

# Check for divergences
divergences = trace_normal.sample_stats["diverging"].sum().values
print(f"\nNumber of divergences: {int(divergences)}")
if int(divergences) == 0:
    print("No divergences detected -- the sampler explored the posterior well.")
else:
    print("WARNING: Divergences detected. Consider reparameterization.")

In [ ]:
# --- Forest plot: compare chains ---
az.plot_forest(trace_normal, var_names=["mu", "sigma"], combined=False,
               hdi_prob=0.94, r_hat=True, ess=True)
plt.title("Forest Plot with R-hat and ESS")
plt.tight_layout()
plt.show()

## 4. Posterior Predictive Checks

We generate replicated datasets from the fitted model and compare them
with the observed data. If the model is adequate, replicated data should
look like the real data.

In [ ]:
# --- Posterior predictive check ---
with normal_model:
    ppc = pm.sample_posterior_predictive(
        trace_normal, random_seed=RANDOM_SEED
    )

az.plot_ppc(ppc, num_pp_samples=100)
plt.title("Posterior Predictive Check")
plt.tight_layout()
plt.show()

## 5. Bayesian Logistic Regression

For binary outcomes $y_i \in \{0, 1\}$:

$$
y_i \sim \text{Bernoulli}(p_i), \quad \text{logit}(p_i) = \alpha + \beta_1 x_{i1} + \beta_2 x_{i2}
$$

In [ ]:
# --- Generate synthetic binary classification data ---
n = 200
true_alpha_lr = -0.5
true_beta1_lr = 1.5
true_beta2_lr = -1.0

X1 = rng.normal(0, 1, size=n)
X2 = rng.normal(0, 1, size=n)

logit_p = true_alpha_lr + true_beta1_lr * X1 + true_beta2_lr * X2
p_true = 1 / (1 + np.exp(-logit_p))
y_bin = rng.binomial(1, p_true)

print(f"Class balance: {y_bin.mean():.1%} positive")

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X1, X2, c=y_bin, cmap="RdYlBu", alpha=0.6,
                     edgecolors="k", linewidths=0.5, s=40)
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("Synthetic Binary Classification Data")
plt.colorbar(scatter, label="Class")
plt.tight_layout()
plt.show()

In [ ]:
# --- Bayesian logistic regression ---
with pm.Model() as logreg_model:
    # Priors
    alpha = pm.Normal("alpha", mu=0, sigma=5)
    beta1 = pm.Normal("beta1", mu=0, sigma=5)
    beta2 = pm.Normal("beta2", mu=0, sigma=5)

    # Linear predictor
    logit_p = alpha + beta1 * X1 + beta2 * X2

    # Likelihood
    y_obs = pm.Bernoulli("y_obs", logit_p=logit_p, observed=y_bin)

    # Sample
    trace_logreg = pm.sample(
        2000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        return_inferencedata=True
    )

print(az.summary(trace_logreg, var_names=["alpha", "beta1", "beta2"]))

In [ ]:
# --- Posterior distributions for logistic regression coefficients ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, var, true_val in zip(
    axes, ["alpha", "beta1", "beta2"],
    [true_alpha_lr, true_beta1_lr, true_beta2_lr]
):
    samples = az.extract(trace_logreg, var_names=[var])[var].values
    az.plot_posterior({var: samples}, ax=ax, ref_val=true_val,
                     hdi_prob=0.94)
    ax.set_title(f"{var} (true = {true_val})")

plt.suptitle("Logistic Regression Posteriors", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# --- Decision boundary from posterior mean ---
alpha_hat = az.extract(trace_logreg)["alpha"].values.mean()
beta1_hat = az.extract(trace_logreg)["beta1"].values.mean()
beta2_hat = az.extract(trace_logreg)["beta2"].values.mean()

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X1, X2, c=y_bin, cmap="RdYlBu", alpha=0.5,
           edgecolors="k", linewidths=0.5, s=40)

# Decision boundary: alpha + beta1*x1 + beta2*x2 = 0
x1_range = np.linspace(X1.min() - 0.5, X1.max() + 0.5, 100)
x2_boundary = -(alpha_hat + beta1_hat * x1_range) / beta2_hat
ax.plot(x1_range, x2_boundary, "k--", lw=2, label="Posterior mean boundary")

# True boundary
x2_true_boundary = -(true_alpha_lr + true_beta1_lr * x1_range) / true_beta2_lr
ax.plot(x1_range, x2_true_boundary, "r-", lw=2, label="True boundary")

ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_ylim(X2.min() - 0.5, X2.max() + 0.5)
ax.set_title("Bayesian Logistic Regression: Decision Boundary")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Bayesian Poisson Regression

For count data $y_i \in \{0, 1, 2, \dots\}$:

$$
y_i \sim \text{Poisson}(\lambda_i), \quad \log(\lambda_i) = \alpha + \beta x_i
$$

In [ ]:
# --- Generate synthetic count data ---
n_pois = 150
true_alpha_p = 1.0
true_beta_p = 0.5

x_pois = rng.uniform(0, 4, size=n_pois)
log_lambda = true_alpha_p + true_beta_p * x_pois
lambda_true = np.exp(log_lambda)
y_counts = rng.poisson(lambda_true)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x_pois, y_counts, alpha=0.5, s=30)
x_sort = np.sort(x_pois)
ax.plot(x_sort, np.exp(true_alpha_p + true_beta_p * x_sort), "r-", lw=2,
        label=r"True $\lambda(x)$")
ax.set_xlabel("x")
ax.set_ylabel("Count")
ax.set_title("Synthetic Poisson Count Data")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- Bayesian Poisson regression ---
with pm.Model() as poisson_model:
    alpha = pm.Normal("alpha", mu=0, sigma=5)
    beta = pm.Normal("beta", mu=0, sigma=5)

    log_lambda = alpha + beta * x_pois
    y_obs = pm.Poisson("y_obs", mu=pm.math.exp(log_lambda), observed=y_counts)

    trace_poisson = pm.sample(
        2000, tune=1000, cores=1, random_seed=RANDOM_SEED,
        return_inferencedata=True
    )

print(az.summary(trace_poisson, var_names=["alpha", "beta"]))

In [ ]:
# --- Posterior predictive check for Poisson regression ---
with poisson_model:
    ppc_poisson = pm.sample_posterior_predictive(
        trace_poisson, random_seed=RANDOM_SEED
    )

az.plot_ppc(ppc_poisson, num_pp_samples=100)
plt.title("Posterior Predictive Check: Poisson Regression")
plt.tight_layout()
plt.show()

In [ ]:
# --- Plot posterior mean curve with uncertainty ---
alpha_samp = az.extract(trace_poisson)["alpha"].values
beta_samp = az.extract(trace_poisson)["beta"].values

x_grid = np.linspace(0, 4.5, 200)
lambda_curves = np.exp(
    alpha_samp[:, None] + beta_samp[:, None] * x_grid[None, :]
)

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(x_pois, y_counts, alpha=0.4, s=25, zorder=5, label="Data")

# Posterior mean
ax.plot(x_grid, lambda_curves.mean(axis=0), "navy", lw=2.5,
        label=r"Posterior mean $\lambda(x)$")

# 94% HDI band
lo = np.percentile(lambda_curves, 3, axis=0)
hi = np.percentile(lambda_curves, 97, axis=0)
ax.fill_between(x_grid, lo, hi, alpha=0.25, color="steelblue",
                label="94% HDI")

# True curve
ax.plot(x_grid, np.exp(true_alpha_p + true_beta_p * x_grid), "r--", lw=2,
        label=r"True $\lambda(x)$")

ax.set_xlabel("x")
ax.set_ylabel(r"$\lambda$ (expected count)")
ax.set_title("Bayesian Poisson Regression: Posterior Rate Function")
ax.legend()
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Metropolis-Hastings** is the foundation of MCMC -- simple but requires tuning.
2. **NUTS** (PyMC default) adapts automatically and explores high-dimensional posteriors efficiently.
3. Always check **R-hat** (< 1.01), **ESS** (> 400), and **divergences** (ideally 0).
4. **Posterior predictive checks** reveal model misfit by comparing simulated data with observed data.
5. **GLMs** (logistic, Poisson) extend the Bayesian framework to non-Normal outcomes with link functions.

---

*Next: Day 3 -- Hierarchical Models*